In [ ]:
# ============================================================
# Cell 1: 安装依赖 + 启动 Ollama (GPU) + 构建 RAG
# ============================================================

import subprocess, os, sys

# --- 检测运行环境 ---
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f'📍 运行环境: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "其他"}')

# --- GPU 检测 ---
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU 已连接: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU! 请在 Kaggle 右侧 Settings → Accelerator 选择 GPU')
    print('   或在 Colab: Runtime → Change runtime type → GPU')

# --- 系统依赖 ---
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg > /dev/null 2>&1

# --- Python 依赖 ---
# 用 whisper tiny/base 降低延迟; faster-whisper 可选但更快
!pip install -q openai-whisper edge-tts aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers

# --- 安装 Ollama ---
!curl -fsSL https://ollama.com/install.sh | sh

# --- 启动 Ollama (GPU 模式) ---
import time, threading

def start_ollama():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    env['OLLAMA_ORIGINS'] = '*'
    # ✅ 关键: 让 Ollama 使用 GPU
    env['OLLAMA_GPU_LAYERS'] = '999'  # 全部层放 GPU
    subprocess.Popen(['ollama', 'serve'], env=env)

threading.Thread(target=start_ollama, daemon=True).start()
print('⏳ 等待 Ollama 启动...')

# 等待 Ollama 就绪 (轮询而非固定 sleep)
import requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 (耗时 {i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ Ollama 启动超时，请检查日志')

# --- 下载模型 (选更小/更快的模型以降低延迟) ---
# llama3.1 (8B) 在 T4 上推理较慢，如需更快可换 qwen2:1.5b 或 phi3:mini
# 将 llama3.1 改为 llama3.2 (3B参数，速度快很多，且依然具备极强的英语理解和指令遵循能力)
print('📥 下载 LLM 模型...')
!ollama pull llama3.2
print('✅ LLM 模型就绪')

# 验证模型是否使用 GPU
!echo '---'; curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; d=json.load(sys.stdin); [print(f'  模型: {m[\"name\"]}') for m in d.get('models',[])]"

# --- RAG 知识库构建 ---
import glob
from pathlib import Path

try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ✅ 自动检测知识库路径 (兼容 Colab / Kaggle)
possible_kb_paths = [
    '/kaggle/input/knowledge-base/RAG/data',      # ← 改这里，匹配你的 dataset 结构
    '/kaggle/input',
    '/content/drive/MyDrive/Colab Notebooks/RAG/data/knowledge_base',
    os.path.join(WORK_DIR, 'knowledge_base'),
]

# Colab: 挂载 Drive
if IS_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
    except:
        pass

def load_hwu_documents(root_dir):
    docs = []
    md_files = glob.glob(os.path.join(root_dir, '**/*.md'), recursive=True)
    print(f'  扫描: {root_dir} → {len(md_files)} 个 .md 文件')
    for fp in md_files:
        try:
            text = Path(fp).read_text(encoding='utf-8')
            lines = text.split('\n', 2)
            if len(lines) < 3:
                continue
            source = lines[0].replace('Source:', '').strip()
            topic = lines[1].replace('Topic:', '').strip()
            content = lines[2].strip()
            docs.append(Document(
                page_content=content,
                metadata={'source': source, 'topic': topic, 'filename': Path(fp).name}
            ))
        except Exception as e:
            print(f'  ⚠️ {fp}: {e}')
    return docs

vector_db = None
for kb_path in possible_kb_paths:
    if os.path.exists(kb_path):
        documents = load_hwu_documents(kb_path)
        if documents:
            chunks = RecursiveCharacterTextSplitter(
                chunk_size=600, chunk_overlap=100
            ).split_documents(documents)
            print(f'  切分为 {len(chunks)} 个片段，构建向量索引...')
            embeddings = HuggingFaceEmbeddings(
                model_name='sentence-transformers/all-MiniLM-L6-v2',
                model_kwargs={'device': 'cuda' if HAS_GPU else 'cpu'}  # ✅ Embedding 也上 GPU
            )
            vector_db = FAISS.from_documents(chunks, embeddings)
            print(f'  ✅ RAG 向量库构建完成 ({len(chunks)} chunks)')
            break

if vector_db is None:
    print('⚠️ 未找到知识库，将以纯 LLM 模式运行')
    print(f'  Kaggle: 请上传 .md 文件到 Dataset')
    print(f'  Colab: 请确认 Google Drive 中有知识库')

print('\n🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。')

In [ ]:
import subprocess, os, time, requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
subprocess.Popen(['ollama', 'serve'])

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 ({i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ 启动失败，查看日志:')
    !cat /tmp/ollama.log 2>/dev/null || echo "无日志"

In [ ]:

# ============================================================
# Cell 2: 实时语音通话 (优化版: 低延迟 + 边生成边说话 + GPU 加速)
# ============================================================

import gradio as gr
import json
import requests
import whisper
import edge_tts
import asyncio
import os
import time
import re
import tempfile
import shutil
import numpy as np
import scipy.io.wavfile as wavfile
import torch
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor

# ✅ 关键修复: 允许嵌套事件循环 (Kaggle/Jupyter 自带 loop)
import nest_asyncio
nest_asyncio.apply()

# ============================================================
# Whisper 加载 (GPU)
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Whisper 设备: {device}')

WHISPER_MODEL_SIZE = 'base'
stt_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)

if device == 'cuda':
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (GPU)')
else:
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (CPU - 会较慢)')

# ============================================================
# 🔥 Warm-up: Pre-heat all components to eliminate cold-start
# ============================================================
print('🔥 预热各组件，消除首次对话冷启动延迟...')

# 1) Warm up Whisper — run a dummy transcription to trigger CUDA kernel compilation
_warmup_audio = np.zeros(16000, dtype=np.float32)  # 1s silent audio
_warmup_path = os.path.join(tempfile.gettempdir(), '_warmup.wav')
wavfile.write(_warmup_path, 16000, (_warmup_audio * 32767).astype(np.int16))
try:
    stt_model.transcribe(_warmup_path, language='en', fp16=(device == 'cuda'))
    print('  ✅ Whisper 预热完成')
except Exception as e:
    print(f'  ⚠️ Whisper 预热跳过: {e}')
os.remove(_warmup_path)

# 2) Warm up Ollama LLM — send a tiny prompt to load model weights into GPU
def _warmup_llm():
    try:
        r = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': 'llama3.2',
                'messages': [{'role': 'user', 'content': 'hi'}],
                'stream': False,
                'options': {'num_predict': 1, 'num_ctx': 128}
            },
            timeout=120
        )
        if r.status_code == 200:
            print('  ✅ LLM 预热完成')
        else:
            print(f'  ⚠️ LLM 预热返回 {r.status_code}')
    except Exception as e:
        print(f'  ⚠️ LLM 预热跳过: {e}')

# 3) Warm up Edge TTS — establish initial WebSocket connection
def _warmup_tts():
    try:
        _tts_path = os.path.join(tempfile.gettempdir(), '_warmup_tts.mp3')
        loop = asyncio.get_event_loop()
        comm = edge_tts.Communicate('hello', 'en-US-EmmaNeural')
        loop.run_until_complete(comm.save(_tts_path))
        if os.path.exists(_tts_path):
            os.remove(_tts_path)
        print('  ✅ TTS 预热完成')
    except Exception as e:
        print(f'  ⚠️ TTS 预热跳过: {e}')

# 4) Warm up RAG embedding — trigger first embedding computation
def _warmup_rag():
    try:
        if 'vector_db' in globals() and vector_db is not None:
            vector_db.as_retriever(search_kwargs={'k': 1}).invoke('hello')
            print('  ✅ RAG 预热完成')
    except Exception as e:
        print(f'  ⚠️ RAG 预热跳过: {e}')

# Run LLM warm-up (heaviest, takes longest)
_warmup_llm()
# Run TTS & RAG warm-up in parallel threads
_t1 = threading.Thread(target=_warmup_tts)
_t2 = threading.Thread(target=_warmup_rag)
_t1.start(); _t2.start()
_t1.join(); _t2.join()

print('🔥 预热全部完成！首次对话将不再有冷启动延迟。\n')

# ============================================================
# VAD 参数
# ============================================================
SILENCE_THRESHOLD = 0.012
SILENCE_DURATION  = 1.2
MIN_SPEECH_DURATION = 0.3

def detect_speech(audio_chunk, sr):
    if audio_chunk is None or len(audio_chunk) == 0:
        return False
    if audio_chunk.dtype == np.int16:
        audio_float = audio_chunk.astype(np.float32) / 32768.0
    elif audio_chunk.dtype == np.int32:
        audio_float = audio_chunk.astype(np.float32) / 2147483648.0
    else:
        audio_float = audio_chunk.astype(np.float32)
    rms = np.sqrt(np.mean(audio_float ** 2))
    return rms > SILENCE_THRESHOLD

def save_buffer_to_wav(audio_buffer, sr):
    if not audio_buffer:
        return None
    combined = np.concatenate(audio_buffer)
    if len(combined) < sr * MIN_SPEECH_DURATION:
        return None
    path = os.path.join(tempfile.gettempdir(), f'voice_{int(time.time()*1000)}.wav')
    if combined.dtype != np.int16:
        if np.max(np.abs(combined)) <= 1.0:
            combined = (combined * 32767).astype(np.int16)
        else:
            combined = combined.astype(np.int16)
    wavfile.write(path, sr, combined)
    return path

# ============================================================
# ASR
# ============================================================
def speech_to_text(audio_path):
    if not audio_path:
        return ''
    try:
        t0 = time.time()
        result = stt_model.transcribe(
            audio_path,
            language='en',
            fp16=(device == 'cuda'),
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
        )
        text = result['text'].strip()
        elapsed = time.time() - t0

        hallucinations = {
            'thank you', 'thanks for watching', 'subscribe',
            'you', 'bye', 'the end', '字幕', '感谢观看',
            '请不吝点赞', '订阅', '谢谢大家', ''
        }
        if text.lower().strip('.!? ') in hallucinations:
            return ''

        print(f'🎤 识别 ({elapsed:.2f}s): {text}')
        return text
    except Exception as e:
        print(f'❌ ASR Error: {e}')
        return ''

# ============================================================
# TTS: 基础 TTS 函数 (nest_asyncio + 重试)
# ============================================================
def _run_tts_sync(text, voice, output_path, retries=2):
    for attempt in range(retries):
        try:
            loop = asyncio.get_event_loop()
            communicate = edge_tts.Communicate(text, voice, rate='+5%')
            loop.run_until_complete(communicate.save(output_path))

            if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
                return True
            else:
                print(f'  ⚠️ TTS 文件太小 (attempt {attempt+1}), 重试...')
        except Exception as e:
            print(f'  ⚠️ TTS attempt {attempt+1} failed: {e}')
            time.sleep(0.5)
    return False

def _clean_text_for_tts(text):
    """Clean text for TTS: remove markdown, URLs, etc."""
    clean = text.split('---')[0]
    clean = re.sub(r'\(https?://[^\)]+\)', '', clean)
    clean = re.sub(r'https?://\S+', '', clean)
    clean = re.sub(r'For more information,?\s*visit:?\s*', '', clean)
    clean = re.sub(r'[*#_>~`\[\](){}|]', '', clean)
    clean = re.sub(r'\\n|\\r', ' ', clean)
    clean = re.sub(r'\n+', '. ', clean)
    clean = re.sub(r'\s{2,}', ' ', clean).strip()
    clean = re.sub(r'\.{2,}', '.', clean)
    return clean

def _detect_tts_voice(text, default_voice='en-US-EmmaNeural'):
    """Auto-detect voice based on text language."""
    has_cjk = bool(re.search(r'[\u4e00-\u9fff]', text))
    return 'zh-CN-XiaoxiaoNeural' if has_cjk else default_voice

# ============================================================
# 🔥 Streaming TTS: 边生成边合成语音 (核心新功能)
# ============================================================

def tts_single_sentence(text, index, voice='en-US-EmmaNeural'):
    """Generate TTS for one sentence. Returns (index, audio_path)."""
    if not text or len(text.strip()) < 2:
        return (index, None)

    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        return (index, None)

    v = _detect_tts_voice(clean, voice)
    path = os.path.join(tempfile.gettempdir(), f'tts_chunk_{index}_{int(time.time()*1000)}.mp3')

    if _run_tts_sync(clean, v, path):
        return (index, path)
    return (index, None)


def concatenate_audio_files(audio_paths):
    """Concatenate multiple MP3 files into one by binary append."""
    valid_paths = [p for p in audio_paths if p and os.path.exists(p) and os.path.getsize(p) > 100]
    if not valid_paths:
        return None
    if len(valid_paths) == 1:
        return valid_paths[0]

    output_path = os.path.join(tempfile.gettempdir(), f'tts_merged_{int(time.time()*1000)}.mp3')
    with open(output_path, 'wb') as outfile:
        for p in valid_paths:
            with open(p, 'rb') as infile:
                outfile.write(infile.read())

    # Clean up individual chunks
    for p in valid_paths:
        try:
            os.remove(p)
        except OSError:
            pass

    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        return output_path
    return None


# Legacy TTS (kept for compatibility, now delegates to helpers)
def text_to_speech(text, voice='en-US-EmmaNeural'):
    if not text or len(text.strip()) < 2:
        return None

    print(f'🔈 TTS 输入: [{text[:100]}]')
    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        print('⚠️ TTS: 清理后文本为空')
        return None

    if len(clean) > 1500:
        cut = clean[:1500].rfind('.')
        if cut > 100:
            clean = clean[:cut+1]
        else:
            clean = clean[:1500]

    v = _detect_tts_voice(clean, voice)
    path = os.path.join(tempfile.gettempdir(), f'tts_{int(time.time()*1000)}.mp3')

    if _run_tts_sync(clean, v, path):
        print(f'🔊 TTS OK: {path} ({len(clean)} chars)')
        return path
    else:
        print('❌ TTS 最终失败')
        return None

# ============================================================
# ✅ 强制 Gradio 每次都重新播放音频
# ============================================================
def make_unique_audio(audio_path):
    if audio_path is None:
        return None
    try:
        unique_path = os.path.join(tempfile.gettempdir(), f'play_{int(time.time()*1000)}.mp3')
        shutil.copy2(audio_path, unique_path)
        return unique_path
    except Exception as e:
        print(f'⚠️ copy error: {e}')
        return audio_path

# ============================================================
# RAG 检索
# ============================================================
def retrieve_context(query):
    if 'vector_db' not in globals() or vector_db is None:
        return ''
    try:
        docs = vector_db.as_retriever(search_kwargs={'k': 3}).invoke(query)
        parts = []
        for i, d in enumerate(docs):
            source_url = d.metadata.get('source', 'N/A')
            topic = d.metadata.get('topic', 'N/A')
            parts.append(f'[Doc {i}] (Source: {source_url} | Topic: {topic}):\n{d.page_content}')
        return '\n\n'.join(parts)
    except Exception as e:
        print(f'RAG Error: {e}')
        return ''

# ============================================================
# 🔥 LLM + Streaming TTS Pipeline (边生成边合成)
# ============================================================

# Minimum characters before flushing a sentence to TTS
_MIN_SENTENCE_LEN = 30

def call_llm_with_streaming_tts(user_text, chat_history):
    """
    Stream LLM output sentence-by-sentence.
    Each completed sentence is immediately submitted to TTS in a thread pool.
    After LLM finishes, collect all TTS audio and concatenate.
    Result: TTS runs IN PARALLEL with LLM generation → much lower latency.
    """
    ctx = retrieve_context(user_text)
    prompt = f"""
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).
    You must be highly empathetic, professional, and supportive.

    Here is the retrieved knowledge from the university database:
    [CONTEXT START]
    {ctx}
    [CONTEXT END]

    You must follow these prioritized guidelines:

    **Priority 1: Crisis Intervention & Mental Health**
    If the user expresses distress, depression, anxiety, or a crisis:
    - Respond with deep empathy and humanistic care.
    - Validate their feelings and prioritize their safety.
    - IMMEDIATELY suggest they contact HWU Student Wellbeing Services.
    - Do NOT say "I apologize, I can only answer...". Be a supportive human.

    **Priority 2: Daily Conversation & Greetings**
    If the user says hello, asks how you are, or engages in basic small talk:
    - Respond warmly and naturally.
    - Gently guide the conversation back to how you can help them with Heriot-Watt University student services.

    **Priority 3: HWU Student Services Queries**
    If the question is about HWU (accommodation, enrollment, campus life, etc.):
    - Provide a RICH, detailed, and structured answer using the [CONTEXT].
    - Expand on the details to make the answer comprehensive rather than overly brief.
    - You MUST include the source URL link from the context at the end. Format: "For more information, visit: [URL]"

    **Priority 4: Out of Scope Queries**
    If the question is completely unrelated to HWU or student services (e.g., coding, math, global news):
    - Politely decline by saying: "I'd love to chat about that, but my expertise is focused on Heriot-Watt University Student Services. How can I help you with your university life today?"

    Output Guidelines:
    - ALWAYS reply in the same language the user is speaking (e.g., if asked in Chinese, reply in natural Chinese).
    - Maintain a conversational and rich tone.
    """

    msgs = [{'role': 'system', 'content': prompt}]
    recent_history = (chat_history or [])[-3:]
    for u, a in recent_history:
        msgs.append({'role': 'user', 'content': u})
        if a:
            msgs.append({'role': 'assistant', 'content': a})
    msgs.append({'role': 'user', 'content': user_text})

    try:
        t0 = time.time()
        resp = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': 'llama3.2',
                'messages': msgs,
                'stream': True,
                'options': {
                    'num_predict': 400,
                    'temperature': 0.6,
                    'num_ctx': 2048,
                }
            },
            timeout=60,
            stream=True
        )

        full_text = ''
        current_buffer = ''
        sentence_index = 0
        tts_executor = ThreadPoolExecutor(max_workers=3)
        tts_futures = []

        for line in resp.iter_lines():
            if line:
                try:
                    chunk = json.loads(line)
                    if 'message' in chunk and 'content' in chunk['message']:
                        token = chunk['message']['content']
                        full_text += token
                        current_buffer += token

                        # Flush buffer to TTS when a sentence boundary is detected
                        if (re.search(r'[.!?。！？]\s*$', current_buffer)
                                and len(current_buffer.strip()) >= _MIN_SENTENCE_LEN):
                            sentence = current_buffer.strip()
                            future = tts_executor.submit(tts_single_sentence, sentence, sentence_index)
                            tts_futures.append((sentence_index, future))
                            print(f'  📝 Chunk {sentence_index}: "{sentence[:60]}..." → TTS submitted')
                            sentence_index += 1
                            current_buffer = ''

                    if chunk.get('done', False):
                        break
                except json.JSONDecodeError:
                    continue

        # Flush remaining text
        if current_buffer.strip() and len(current_buffer.strip()) >= 2:
            future = tts_executor.submit(tts_single_sentence, current_buffer.strip(), sentence_index)
            tts_futures.append((sentence_index, future))
            print(f'  📝 Chunk {sentence_index} (final): "{current_buffer.strip()[:60]}..."')

        llm_elapsed = time.time() - t0
        print(f'🤖 LLM done ({llm_elapsed:.1f}s, {sentence_index + 1} chunks), waiting for remaining TTS...')

        # Collect TTS results IN ORDER
        tts_futures.sort(key=lambda x: x[0])
        audio_paths = []
        for idx, future in tts_futures:
            try:
                _, audio_path = future.result(timeout=30)
                if audio_path:
                    audio_paths.append(audio_path)
            except Exception as e:
                print(f'  ⚠️ TTS chunk {idx} failed: {e}')

        tts_executor.shutdown(wait=False)

        # Concatenate all chunks into one audio file
        final_audio = concatenate_audio_files(audio_paths)

        total_elapsed = time.time() - t0
        print(f'⏱️ Streaming pipeline total: {total_elapsed:.1f}s (LLM={llm_elapsed:.1f}s, TTS overlap saved time!)')

        return (full_text if full_text else 'Sorry, I could not generate a response.'), final_audio

    except requests.Timeout:
        return 'Sorry, the response timed out. Please try again.', None
    except Exception as e:
        print(f'❌ Streaming LLM+TTS Error: {e}')
        traceback.print_exc()
        return f'Sorry, service error: {e}', None


# ============================================================
# 核心: 流式音频 VAD 回调 (使用 streaming TTS pipeline)
# ============================================================
def process_streaming_audio(audio_chunk, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }

    if state_data.get('processing', False):
        return state_data, state_data.get('chat_history', []), gr.update(), '🔄 AI is processing...'

    if audio_chunk is None:
        return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Waiting for you to speak...'

    sr, audio_data = audio_chunk
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    now = time.time()
    is_voice = detect_speech(audio_data, sr)

    if is_voice:
        state_data['audio_buffer'].append(audio_data)
        state_data['is_speaking'] = True
        state_data['speech_detected'] = True
        state_data['silence_start'] = None
        return state_data, state_data.get('chat_history', []), gr.update(), '🗣️ Listening...'
    else:
        if state_data['speech_detected']:
            state_data['audio_buffer'].append(audio_data)
            if state_data['silence_start'] is None:
                state_data['silence_start'] = now

            silence_elapsed = now - state_data['silence_start']

            if silence_elapsed >= SILENCE_DURATION:
                state_data['processing'] = True
                print(f'\n⏹️ 静音 {silence_elapsed:.1f}s，开始处理...')

                wav_path = save_buffer_to_wav(state_data['audio_buffer'], sr)
                state_data['audio_buffer'] = []
                state_data['is_speaking'] = False
                state_data['speech_detected'] = False
                state_data['silence_start'] = None

                if wav_path is None:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Audio too short, please try again...'

                pipeline_start = time.time()

                user_text = speech_to_text(wav_path)
                if not user_text:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Could not hear clearly, please repeat...'

                # 🔥 Use streaming TTS pipeline (LLM + TTS run in parallel)
                ai_text, audio_out = call_llm_with_streaming_tts(user_text, state_data.get('chat_history', []))

                total_latency = time.time() - pipeline_start
                print(f'⏱️ 总延迟: {total_latency:.1f}s (ASR + LLM + Streaming TTS)')

                history = state_data.get('chat_history', [])
                history.append((user_text, ai_text))
                state_data['chat_history'] = history
                state_data['processing'] = False

                return state_data, history, make_unique_audio(audio_out), f'🎙️ Done ({total_latency:.1f}s), keep speaking...'
            else:
                remaining = SILENCE_DURATION - silence_elapsed
                return state_data, state_data.get('chat_history', []), gr.update(), f'🤫 Paused... ({remaining:.1f}s)'
        else:
            return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Waiting for you to speak...'

# ============================================================
# 文字输入 (备用, 也使用 streaming TTS)
# ============================================================
def handle_text_input(user_text, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }
    if not user_text or not user_text.strip():
        return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Waiting for you to speak...', ''

    # 🔥 Use streaming TTS pipeline
    ai_text, audio_out = call_llm_with_streaming_tts(user_text, state_data.get('chat_history', []))
    history = state_data.get('chat_history', [])
    history.append((user_text, ai_text))
    state_data['chat_history'] = history

    return state_data, history, make_unique_audio(audio_out), '🎙️ AI replied, you can speak now...', ''


# ============================================================
# Gradio 界面 (完美对齐 + 渐变标题 + 深夜模式)
# ============================================================

custom_css = """
/* 隐藏 Gradio 顶部的默认留白和页脚水印 */
footer { display: none !important; }
.gradio-container { border: none !important; background: transparent !important; }

/* 基础背景 (白天模式) - 保留你喜欢的弥散渐变 */
body, .gradio-container {
    background-color: #fafafa !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(244, 114, 182, 0.25) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(167, 139, 250, 0.15) 0%, transparent 50%) !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    transition: background-color 0.4s ease, background-image 0.4s ease !important;
}

/* 深夜模式背景 - 色调调暗，光晕变为暗紫红和深蓝 */
body.dark-mode, body.dark-mode .gradio-container {
    background-color: #0f172a !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(190, 24, 93, 0.2) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(76, 29, 149, 0.2) 0%, transparent 50%) !important;
}

/* 玻璃质感圆角卡片 */
.ui-card {
    background: rgba(255, 255, 255, 0.85) !important;
    border-radius: 24px !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.08) !important;
    border: 1px solid rgba(255, 255, 255, 1) !important;
    padding: 20px !important;
    margin-bottom: 15px !important;
    transition: background 0.4s ease, border-color 0.4s ease !important;
}
/* 深夜模式的卡片变色 */
body.dark-mode .ui-card {
    background: rgba(30, 41, 59, 0.85) !important;
    border: 1px solid rgba(255, 255, 255, 0.05) !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.4) !important;
}

/* 动态文字颜色 (供普通文本使用) */
.dyn-text { color: #475569 !important; transition: color 0.4s ease; }
body.dark-mode .dyn-text { color: #cbd5e1 !important; }
.dyn-subtext { color: #64748b !important; }
body.dark-mode .dyn-subtext { color: #94a3b8 !important; }

/* =========================================
   要求1：标题使用图2中的紫色渐变风格和大小 
   ========================================= */
.gradient-title {
    font-size: 2.8rem !important;
    font-weight: 800 !important;
    /* 完美复刻图中紫粉色渐变 */
    background: linear-gradient(90deg, #c026d3, #8b5cf6) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    margin-bottom: 0 !important;
}
/* 深夜模式下让渐变更亮一点点 */
body.dark-mode .gradient-title { background: linear-gradient(90deg, #e879f9, #a78bfa) !important; -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important; }

.subtitle {
    font-size: 1.2rem !important;
    font-weight: 600 !important;
    color: #334155 !important;
    margin-top: 5px !important;
}
body.dark-mode .subtitle { color: #e2e8f0 !important; }

/* =========================================
   要求2：文本框和发送按钮绝对水平对齐
   ========================================= */
.input-row {
    display: flex !important;
    align-items: center !important; /* 强制垂直居中对齐 */
    gap: 12px !important;
    padding: 10px 15px !important;
}
/* 去除 Textbox 自带的外边框和内边距，防止飘上去 */
.input-box { flex-grow: 1 !important; margin: 0 !important; }
.input-box > div { border: none !important; box-shadow: none !important; background: transparent !important; }
.input-box textarea {
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    padding: 12px 15px !important;
    font-size: 15px !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .input-box textarea { background: #1e293b !important; border-color: #475569 !important; color: #f8fafc !important; }
.input-box textarea:focus { border-color: #8b5cf6 !important; outline: none !important; }

/* 发送按钮统一样式 */
.send-btn {
    margin: 0 !important;
    height: 48px !important; /* 锁死高度与文本框一致 */
    border-radius: 12px !important;
    background: linear-gradient(90deg, #ec4899, #8b5cf6) !important;
    color: white !important;
    font-weight: 600 !important;
    border: none !important;
    min-width: 140px !important;
    transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}
.send-btn:hover { transform: translateY(-2px); box-shadow: 0 8px 20px -5px rgba(236, 72, 153, 0.4) !important; }

/* 主题切换按钮样式 */
.theme-btn {
    border-radius: 12px !important;
    background: transparent !important;
    border: 1px solid #cbd5e1 !important;
    color: #475569 !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .theme-btn { border-color: #475569 !important; color: #cbd5e1 !important; background: rgba(30, 41, 59, 0.5) !important; }

/* 对话框气泡优化 */
#chatbot { background: transparent !important; border: none !important; }
#chatbot .message { border-radius: 16px !important; }
"""

theme = gr.themes.Soft(
    primary_hue="pink", 
    secondary_hue="purple",
    neutral_hue="slate"
)

with gr.Blocks(title='HWU Voice Call', css=custom_css, theme=theme) as demo:
    
    # 顶部区域
    with gr.Row():
        with gr.Column(scale=9):
            # 标题区：使用 HTML 精准套用渐变色 CSS
            gr.HTML(f"""
                <div style="text-align: center; margin-top: 2rem; margin-bottom: 2.5rem;">
                    <h1 class="gradient-title">HWU AI Student Services</h1>
                    <h2 class="subtitle">Live Voice Call</h2>
                    <p class="dyn-subtext" style="font-size: 1.05rem; max-width: 650px; margin: 1.5rem auto 0; line-height: 1.6;">
                        <strong>Instructions:</strong> Click the microphone to start the call, speak directly, and the AI will automatically reply after a {SILENCE_DURATION}s pause.<br>
                        <span style="font-size: 0.85em; opacity: 0.8;">Environment: GPU={torch.cuda.is_available()} | Whisper={WHISPER_MODEL_SIZE} | Device={device} | 🔥 Streaming TTS enabled</span>
                    </p>
                </div>
            """)
        # 要求3：切换深色模式的按钮
        with gr.Column(scale=1):
            theme_btn = gr.Button("🌓 Theme", elem_classes="theme-btn")
            theme_btn.click(None, None, None, js="""
                function() {
                    document.body.classList.toggle('dark-mode');
                }
            """)

    call_data = gr.State(value={
        'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
        'speech_detected': False, 'chat_history': [], 'processing': False
    })

    # 第一行：对话记录 + 麦克风控制区
    with gr.Row():
        with gr.Column(scale=3, elem_classes="ui-card"):
            chatbot = gr.Chatbot(label='Conversation History', height=420, elem_id="chatbot")
            
        with gr.Column(scale=1, elem_classes="ui-card"):
            status = gr.Textbox(value='🔴 Not Started', label='Call Status', interactive=False)
            audio_player = gr.Audio(label='🔊 AI Response', autoplay=True, interactive=False)
            
            gr.Markdown('<hr style="margin: 15px 0; border-color: #e2e8f0;">')
            
            # 原生麦克风组件，保留不变
            mic = gr.Audio(
                sources=['microphone'], streaming=True,
                label='🎤 Microphone (Click to start)'
            )
            clear_btn = gr.Button('🗑️ Clear Conversation', variant='stop')

    # 要求2：第二行：文字输入区（完全对齐）
    with gr.Row(elem_classes="ui-card input-row"):
        text_input = gr.Textbox(
            show_label=False, # 隐藏label是解决错位飘起的关键
            container=False,  # 剥离外层容器，适应flex布局
            placeholder='Or type your questions here...',
            scale=4,
            elem_classes="input-box"
        )
        send_btn = gr.Button('Send Message', scale=1, elem_classes="send-btn")

    # --- 核心事件绑定 ---
    mic.stream(
        fn=process_streaming_audio,
        inputs=[mic, call_data],
        outputs=[call_data, chatbot, audio_player, status]
    )
    send_btn.click(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )
    text_input.submit(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )

    def clear_all():
        return (
            {'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
             'speech_detected': False, 'chat_history': [], 'processing': False},
            [], None, '🎙️ Session Cleared'
        )
    clear_btn.click(fn=clear_all, outputs=[call_data, chatbot, audio_player, status])

print('\n🚀 启动实时语音通话 (Streaming TTS 已启用)...')
demo.launch(debug=True, share=True)
